# 70 Auxiliary MediaPipe nasion route

Compares the automatic MediaPipe nasion detector to the manually picked nasion across the seven thesis subjects. Feeds Table 4.8 (confidence + distance) of the thesis.

**BRANCH REQUIRED.** The auto-nasion code lives on the `auto-detection-pipeline` branch. Run `git checkout auto-detection-pipeline` before executing, then switch back to `main` afterwards.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_helpers import (
    SUBJECTS, subject_paths, load_raw, load_anon, load_landmarks,
    available_subjects, missing_report, run_pipeline,
)

import numpy as np
import pandas as pd

OUT_DIR = pathlib.Path('thesis_results_out')
OUT_DIR.mkdir(exist_ok=True)

## 1. Import the auto-nasion detector

Only succeeds on the `auto-detection-pipeline` branch; on `main` the import fails and the notebook reports the missing branch instead of running.

In [ ]:
try:

    from cedalion.geometry.photogrammetry.anonymization.nasion_detector import (

        detect_nasion_auto,

    )

    HAS_AUTO = True

except ImportError as err:

    HAS_AUTO = False

    print('Auto-nasion detector is not on this branch:')

    print(f'  {err}')

    print('Run: git checkout auto-detection-pipeline')

## 2. Per-subject comparison

In [ ]:
rows = []

if HAS_AUTO:

    for n in SUBJECTS:

        paths = subject_paths(n)

        if not (paths.raw_exists and paths.landmarks_exist):

            print(f'skipping Subject{n}: missing raw or landmarks')

            continue

        surface_raw = load_raw(n)

        landmarks_raw = load_landmarks(n)

        Nz_manual = landmarks_raw.sel(label='Nz').pint.dequantify().values

        result = detect_nasion_auto(surface_raw)

        conf = float(result.confidence)

        rows.append({

            'subject': n,

            'confidence': conf,

            'distance_to_manual_mm': float(np.linalg.norm(Nz_manual - result.position)),

            'outcome': 'success' if conf >= 0.3 else 'fallback',

        })

        print(rows[-1])

## 3. Summary table

In [ ]:
df = pd.DataFrame(rows).sort_values('subject').reset_index(drop=True) if rows else pd.DataFrame()

df

## 4. Save CSV

In [ ]:
if len(df):

    out = OUT_DIR / 'auxiliary_nasion.csv'

    df.to_csv(out, index=False)

    print(f'Wrote {out}')